In [1]:
from xgboost.sklearn import XGBRegressor
import xgboost as xgb
import pandas as pd
import numpy as np

features = [
    "wind_speed",
    # "track_distance",
    "rainfall_max_24h",
    "N_events_5_years_weighted",
    # "N_events_5_years",
    "population",
    "coast_length",
    "with_coast",
    "mean_elev",
    "mean_slope",
    "mean_rug",
    "urban",
    "rural",
    "water",
    "storm_tide_rp_0010",
    "landslide_risk_sum",
    # "flood_risk",
    # "shdi",
    # "perc_affected_pop_grid_region"
    ]

xgb_params_classifier = {
        # the same
        "booster": "gbtree",
        "colsample_bytree": 0.8,
        "gamma": 0.5,
        "learning_rate": 0.01,
        "max_depth": 4,
        "min_child_weight": 1,
        "n_estimators": 100,
        "subsample": 0.8,
        "verbosity": 0,
        "random_state": 0,
        # binary classification
        "objective": "binary:logistic",  # outputs probability between 0 and 1
        "eval_metric": "logloss"         # binary classification loss
    }

xgb_params_regressor = {
        "booster": "gbtree",
        "colsample_bytree": 0.8,
        "gamma": 0.5,
        "learning_rate": 0.01,
        "max_depth": 4,
        "min_child_weight": 1,
        "n_estimators": 100,
        # "early_stopping_rounds": 10,
        "objective": "reg:squarederror",  # squared error
        "eval_metric": "rmse",            # RMSE metric
        "subsample": 0.8,
        "verbosity": 0,
        "random_state": 0,
    }

In [2]:
def first_stage_classifier_proba(df_test, df_train, features, xgb_params, target_name):

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(df_train[features], df_train[target_name])

    y_proba = model.predict_proba(df_test[features])[:, 1]  # prob of class 1
    y_pred_default = (y_proba >= 0.5).astype(int)

    df_result = df_test.copy()
    df_result["predicted_proba"] = y_proba
    df_result["predicted_bin"] = y_pred_default

    return df_result

def second_stage_regressor(df_train, df_test, features, target_name, xgb_params):
    """
    Second-stage regression after first-stage classification.
    Predicts target for rows with predicted_bin == 1, sets 0 for predicted_bin == 0.

    Returns df_test with 'predicted' column.
    """
    try:
        # Rows predicted as 1 by first stage
        df_stage2 = df_test[df_test["predicted_bin"] == 1].copy()
        X_train, y_train = df_train[features], df_train[target_name]
        X_test_stage2 = df_stage2[features]

        model = XGBRegressor(**xgb_params)
        model.fit(X_train, y_train, eval_set=[(X_train, y_train)], verbose=False)

        df_stage2["predicted"] = model.predict(X_test_stage2)

        # Rows predicted as 0 by first stage
        df_stage2_zero = df_test[df_test["predicted_bin"] == 0].copy()
        df_stage2_zero["predicted"] = 0

        # Concatenate and sort to original order
        df_test_full = pd.concat([df_stage2, df_stage2_zero], axis=0).sort_index()

        return df_test_full

    except Exception as e:
        print(f"Second stage training failed: {e}")
        return None

from xgboost.sklearn import XGBRegressor
from xgboost import XGBClassifier

def prepare_and_train_two_stage_models(
        df,
        features=features,
        u1=3,  # multiplier for majority class in stage 1
        u2=3,  # multiplier for majority class in stage 2
        target_regressor="perc_affected_pop_grid_region",
        xgb_params_classifier=xgb_params_classifier,
        xgb_params_regressor=xgb_params_regressor,
        save_path_classifier="/data/big/fmoss/data/past_forecasts/model_first_stage_model.json",
        save_path_regressor="/data/big/fmoss/data/past_forecasts/second_stage_model.json"
):
    """
    Build training sets for both stages from df,
    train and save classifier and regressor.
    """
    # 1️⃣ Stage 1: Classifier training set
    df_train = df.copy()

    # Create binary target for stage 1
    df_train["reported_bin"] = (df_train[target_regressor] > 0).astype(int)

    minority_size = df_train['reported_bin'].sum()
    majority_size = int(u1 * minority_size)

    df_train_balanced_1stage = (
        df_train.groupby('reported_bin', group_keys=False)
        .apply(lambda x: x.sample(minority_size if x.name == 1 else majority_size, random_state=42))
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    # 2️⃣ Stage 2: Regressor training set
    df_train_high = df_train[df_train[target_regressor] != 0].copy()
    df_train_high["impact_high"] = (df_train_high[target_regressor] >= 15).astype(int)

    minority_size = df_train_high['impact_high'].sum()
    majority_size = int(u2 * minority_size)

    df_train_balanced_2stage = (
        df_train_high.groupby('impact_high', group_keys=False)
        .apply(lambda x: x.sample(minority_size if x.name == 1 else majority_size, random_state=42))
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    # 3️⃣ Train + Save Models
    clf = XGBClassifier(**xgb_params_classifier)
    clf.fit(df_train_balanced_1stage[features], df_train_balanced_1stage["reported_bin"])
    clf.save_model(save_path_classifier)

    reg = XGBRegressor(**xgb_params_regressor)
    reg.fit(df_train_balanced_2stage[features], df_train_balanced_2stage[target_regressor])
    reg.save_model(save_path_regressor)

    print(f"✅ Saved models:\n - Classifier: {save_path_classifier}\n - Regressor: {save_path_regressor}")

    return clf, reg, df_train_balanced_1stage, df_train_balanced_2stage


In [2]:
# Load data
df = pd.read_parquet("/data/big/fmoss/data/model_input_dataset/training_dataset_weighted_N_events.parquet")

# People affected > 100 and subnational levels
events_to_consider_c1 = df[df["Total Affected"] > 100]["DisNo."].unique()
events_to_consider_c2 = df[df["level"] != "ADM0"]["DisNo."].unique()
# More constraints regarding subnational reported events
proportions = (
    df[df["level"] != "ADM0"]
    .assign(is_affected=lambda x: x["Total Affected"] > 0)
    .groupby(["DisNo.", "GID_0", "GID_1"])["is_affected"].max()
    .groupby(["DisNo.", "GID_0"]).mean()
    .reset_index(name="proportion_affected_gid1")
)
events_to_consider_c3 = proportions[proportions.proportion_affected_gid1 < 1]["DisNo."].unique()
events_to_consider = list(set(events_to_consider_c1) & set(events_to_consider_c2) & set(events_to_consider_c3))
df = df[df["DisNo."].isin(events_to_consider)]


In [5]:
disno = "2017-0381-ATG"
sid_disno = "2017242N16333"
iso3_disno = "ATG"

In [8]:
print(len(df[(df["sid"] != sid_disno) & (df["iso3"] != iso3_disno) & (df["DisNo."] != disno)].drop_duplicates()))
print(len(df[df["DisNo."] != disno].drop_duplicates()))

16998656
17116830


In [ ]:
testing_data = pd.read_parquet("/data/big/fmoss/data/past_forecasts/testing_dataset_kms.parquet")
events_testing = testing_data["DisNo."].unique()
training_data = df[~df["DisNo."].isin(events_testing)]

In [6]:
clf, reg, df_train_balanced_1stage, df_train_balanced_2stage = prepare_and_train_two_stage_models(training_data)

/tmp/ipykernel_2306596/1027062965.py:75: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(minority_size if x.name == 1 else majority_size, random_state=42))
/tmp/ipykernel_2306596/1027062965.py:89: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(minority_size if x.name == 1 else majority_size, random_state=42))


✅ Saved models:
 - Classifier: /data/big/fmoss/data/past_forecasts/model_first_stage_model.json
 - Regressor: /data/big/fmoss/data/past_forecasts/second_stage_model.json


In [ ]:
# # Load data
# model_input = pd.read_parquet("/data/big/fmoss/data/model_input_dataset/training_dataset_weighted_N_events.parquet")
# testing_data = pd.read_parquet("/data/big/fmoss/data/past_forecasts/testing_dataset_kms.parquet")

# # Create training set
# events_to_consider = testing_data["DisNo."].unique()
# training_data = model_input[~model_input["DisNo."].isin(events_to_consider)]

In [ ]:
# _ = train_model(training_data, features)